In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from IPython.display import display
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR   = CONFIGS['filepaths']['splits']
WEIGHTSDIR  = CONFIGS['filepaths']['weights']
MODELSDIR   = CONFIGS['filepaths']['models']
PREDSDIR    = CONFIGS['filepaths']['predictions']
FIELDVARS   = CONFIGS['experiments']['nn']['runs']['nn_gauss']['fieldvars']
SEEDS       = CONFIGS['experiments']['nn']['seeds']
EQUATIONS   = CONFIGS['experiments']['sr']['optimizedeqs']
REFNAME     = 'nn_gauss'
MODELINFO   = {**{name:(eq['description'],eq['color'],'-') for name,eq in EQUATIONS.items()},
               REFNAME:(CONFIGS['experiments']['nn']['runs'][REFNAME]['description'],'gray','--')}
OBSINFO     = {'era5':('ERA5',dict(color='k',marker='o',markersize=3,linestyle='none')),
               'imerg':('IMERG',dict(color='k',marker='s',markersize=3,markerfacecolor='none',linestyle='none'))}
LABELS      = {name:info[0] for name,info in {**OBSINFO,**MODELINFO}.items()}
LATRANGE    = CONFIGS['domain']['latrange']
LONRANGE    = CONFIGS['domain']['lonrange']
SPLIT       = 'test'
LANDFRAC    = 0.5
NHOURBINS   = 8
TABLEMODELS = ['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']
REGIONS     = {'Ocean':'#539ED4','Land':'#D42028'}
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
REGISTRY    = {row['name']:json.loads(row['constants']) for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

In [ ]:
def to_precip(exponent):
    return np.maximum(np.expm1(exponent),0.0)

def bin_by_hour(z,mask):
    counts = np.bincount(hourbin[mask],minlength=NHOURBINS)
    sums   = np.bincount(hourbin[mask],weights=z[mask],minlength=NHOURBINS)
    return sums/np.maximum(counts,1)

def calc_peak_bin(z):
    ngrid     = nlat*nlon
    gridindex = np.broadcast_to(np.arange(ngrid),(ntime,ngrid)).ravel()
    index     = hourbin[finite]*ngrid+gridindex[finite]
    counts    = np.bincount(index,minlength=NHOURBINS*ngrid).reshape(NHOURBINS,ngrid)
    sums      = np.bincount(index,weights=z[finite],minlength=NHOURBINS*ngrid).reshape(NHOURBINS,ngrid)
    composite = np.where(counts>0,sums/np.maximum(counts,1),np.nan)
    filled    = np.where(np.isfinite(composite),composite,-np.inf)
    valid     = np.isfinite(composite).all(axis=0)&(filled.max(axis=0)>0)
    return composite,np.where(valid,filled.argmax(axis=0),np.nan)

def format_map(ax,title):
    ax.format(title=title,coast=True,lonlim=LONRANGE,lonlines=[65,75,85],lonlabels='b',latlim=LATRANGE,latlines=[10,15,20],latlabels='l',grid=False)

In [ ]:
c3,c4,c5 = (REGISTRY['sr_atm_eq'][name] for name in ['c3','c4','c5'])
lam      = STATS['tp_std']*c3/STATS['thetae_std']**3
gamma    = c4*STATS['thetae_std']/STATS['thetaestar_std']
thetac   = STATS['thetae_mean']-gamma*STATS['thetaestar_mean']+c5*STATS['thetae_std']
kappa    = STATS['thetae_std']/STATS['rh_std']
sfc,pc   = REGISTRY['sr_sfc_eq'],REGISTRY['sr_all_pc_eq']

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = (ds.sizes[dim] for dim in ['time','lat','lon'])
    lat,lon,dsig    = ds['lat'].values,ds['lon'].values,ds['dsig'].values
    hours  = ds['time'].dt.hour.values
    fields = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,dsig.size) for name in FIELDVARS],axis=1)
    tp,pr,shf,lhf = (ds[name].transpose('time','lat','lon').values.ravel() for name in ['tp','pr','shf','lhf'])
    lf     = xr.broadcast(ds['lf'],ds['tp'])[0].transpose('time','lat','lon').values.ravel().astype(np.float64)
predtp = {}
for name in MODELINFO:
    path = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if os.path.exists(path):
        with xr.open_dataset(path) as ds:
            da = ds['tp'].load()
        predtp[name] = (da.mean('seed') if 'seed' in da.dims else da).squeeze().transpose('time','lat','lon').values.ravel()
print(f"Loaded predictions: {', '.join(LABELS[name] for name in predtp)}")
kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds['k'].values)

integrals   = dict(zip(FIELDVARS,(fields*np.mean(kernels,axis=0)[None,:,:]*dsig[None,None,:]).sum(axis=2).T))
moisture    = kappa*(integrals['rh']-STATS['rh_mean'])
instability = integrals['thetae']-gamma*integrals['thetaestar']-thetac
thetaeanom  = integrals['thetae']-STATS['thetae_mean']
atmexp      = lam*np.maximum(moisture,instability)**3
surface     = STATS['tp_std']*sfc['c6']/STATS['shf_std']*(sfc['c7']-lf)*(shf-STATS['shf_mean'])+STATS['tp_std']*sfc['c8']/STATS['lhf_std']*(lhf-STATS['lhf_mean'])
correction  = STATS['tp_std']/STATS['thetae_std']*((pc['c13']-lf)**3+(1-pc['c13'])**3)*thetaeanom+STATS['tp_std']*pc['c12']/STATS['shf_std']*(pc['c13']-lf)**3*(shf-STATS['shf_mean'])
obstp       = {'era5':tp,'imerg':3.0*pr}
alltp       = {**obstp,**predtp}
finite      = np.isfinite(moisture)&np.isfinite(instability)&np.isfinite(surface)&np.isfinite(correction)&np.all([np.isfinite(values) for values in alltp.values()],axis=0)
for name,exponent in [('sr_atm_eq',atmexp),('sr_sfc_eq',atmexp+surface),('sr_all_pc_eq',atmexp+correction)]:
    print(f'Max difference from saved {LABELS[name]} predictions: {np.nanmax(np.abs(to_precip(exponent)[finite]-predtp[name][finite])):.2e} mm')

shape       = (ntime,nlat,nlon)
localhour   = (hours[:,None,None]+lon[None,None,:]/15.0)%24
hourbin     = (np.broadcast_to(localhour,shape).ravel()//(24/NHOURBINS)).astype(int)
hourcenters = (np.arange(NHOURBINS)+0.5)*24/NHOURBINS
regionmasks = {'Ocean':finite&(lf<LANDFRAC),'Land':finite&(lf>=LANDFRAC)}
lfgrid      = lf.reshape(shape)[0].ravel()
gridmasks   = {'Ocean':lfgrid<LANDFRAC,'Land':lfgrid>=LANDFRAC}
composites,peakbins = {},{}
for name,values in alltp.items():
    composites[name],peakbins[name] = calc_peak_bin(values)
cycles      = {(name,region):bin_by_hour(values,regionmasks[region]) for name,values in alltp.items() for region in REGIONS}
amplitudes  = {name:(np.nanmax(composite,axis=0)-np.nanmin(composite,axis=0)).reshape(nlat,nlon) for name,composite in composites.items()}
amplim      = np.nanpercentile(np.concatenate([amplitudes[name].ravel() for name in OBSINFO]),98)
termcycles  = {(term,region):bin_by_hour(values,regionmasks[region]) for region in REGIONS
               for term,values in [('M',moisture),('I',instability),('dominance',(moisture>instability).astype(float)),('SHF',shf),('LHF',lhf),('S',surface),('C',correction)]}
for region in REGIONS:
    print(f"{region}: diurnal range of mean M = {np.ptp(termcycles['M',region]):.2f} K, mean I = {np.ptp(termcycles['I',region]):.2f} K")

In [ ]:
rows = []
for ref in OBSINFO:
    for region in REGIONS:
        refcycle = cycles[ref,region]
        refpeak  = peakbins[ref][gridmasks[region]]
        for name in alltp:
            if name==ref:
                continue
            cycle    = cycles[name,region]
            peak     = peakbins[name][gridmasks[region]]
            valid    = np.isfinite(peak)&np.isfinite(refpeak)
            distance = np.abs(peak[valid]-refpeak[valid])%NHOURBINS
            rows.append({'Reference':LABELS[ref],
                         'Region':region,
                         'Model':LABELS[name],
                         'Mean Bias (mm)':cycle.mean()-refcycle.mean(),
                         'Amplitude Ratio':np.ptp(cycle)/np.ptp(refcycle),
                         'Normalized Amplitude Ratio':(np.ptp(cycle)/cycle.mean())/(np.ptp(refcycle)/refcycle.mean()),
                         'Correlation':np.corrcoef(cycle,refcycle)[0,1],
                         'Peak Hour (h)':hourcenters[np.argmax(cycle)],
                         'Reference Peak Hour (h)':hourcenters[np.argmax(refcycle)],
                         'Grid Peak Within 3 h (%)':100*np.mean(np.minimum(distance,NHOURBINS-distance)<=1)})
display(pd.DataFrame(rows).style.hide(axis='index').format(precision=2))

rows = []
for name in ['era5',*TABLEMODELS]:
    row   = {'Model':LABELS[name]}
    model = name!='era5'
    for region in REGIONS:
        mask = regionmasks[region]
        row[(region,'R2')]    = 1-np.sum((alltp[name][mask]-tp[mask])**2)/np.sum((tp[mask]-tp[mask].mean())**2) if model else np.nan
        row[(region,'Range')] = 100*np.ptp(cycles[name,region])/np.ptp(cycles['era5',region])
        row[(region,'r')]     = np.corrcoef(cycles[name,region],cycles['era5',region])[0,1] if model else np.nan
        row[(region,'Peak')]  = hourcenters[np.argmax(cycles[name,region])]
    rows.append(row)
table = pd.DataFrame(rows).set_index('Model')
table.columns = pd.MultiIndex.from_tuples(table.columns)
display(table.style.format(lambda value:'--' if np.isnan(value) else f'{value:.3f}'))
formats = {'R2':'{:.3f}','Range':'{:.0f}','r':'{:.2f}'}
for model,values in table.iterrows():
    entries = [f'{int(value-12/NHOURBINS):02d}--{int(value+12/NHOURBINS):02d}' if column[1]=='Peak' else ('--' if np.isnan(value) else formats[column[1]].format(value)) for column,value in values.items()]
    print(' & '.join([model,*entries])+' \\\\')

In [ ]:
handles  = [Line2D([],[],label=label,**style) for label,style in OBSINFO.values()]
handles += [Line2D([],[],color=MODELINFO[name][1],linestyle=MODELINFO[name][2],linewidth=1,label=MODELINFO[name][0]) for name in predtp]

fig,axs = pplt.subplots(nrows=2,ncols=2,refwidth=2.5,refheight=1.75,share=False)
for col,region in enumerate(REGIONS):
    for name,(_,style) in OBSINFO.items():
        axs[0,col].plot(hourcenters,cycles[name,region],zorder=3,**style)
        axs[1,col].plot(hourcenters,cycles[name,region]/cycles[name,region].mean(),zorder=3,**style)
    for name in predtp:
        _,color,linestyle = MODELINFO[name]
        axs[0,col].plot(hourcenters,cycles[name,region],color=color,linestyle=linestyle,linewidth=1)
        axs[1,col].plot(hourcenters,cycles[name,region]/cycles[name,region].mean(),color=color,linestyle=linestyle,linewidth=1)
    axs[0,col].format(title=f'Diurnal Cycle Over {region}')
    axs[1,col].format(title=f'Normalized Diurnal Cycle Over {region}')
axs[0,0].format(ylabel='Total Precipitation (mm)')
axs[1,0].format(ylabel='Precipitation / Diurnal Mean')
fig.legend(handles,loc='b',ncols=4,frame=False)
axs.format(xlabel='Local Solar Time at Window Start (h)',xlim=(0,24),xticks=6,abc=True,titleloc='l')
pplt.show()

In [ ]:
fig,axs = pplt.subplots(nrows=2,ncols=4,proj='cyl',refwidth=2.2,wspace=1.5,hspace=3)
for ax,name in zip(axs,alltp):
    m = ax.pcolormesh(lon,lat,((peakbins[name]+0.5)*24/NHOURBINS).reshape(nlat,nlon),cmap='twilight',vmin=0,vmax=24,levels=NHOURBINS)
    format_map(ax,LABELS[name])
fig.colorbar(m,loc='b',ticks=3,label='Local Hour of Peak Precipitation at Window Start (h)')
axs.format(abc=True,titleloc='l')
pplt.show()

In [ ]:
fig,axs = pplt.subplots(nrows=2,ncols=4,proj='cyl',refwidth=2.2,wspace=1.5,hspace=3)
for ax,name in zip(axs,alltp):
    m = ax.pcolormesh(lon,lat,amplitudes[name],cmap='Blues',vmin=0,vmax=amplim,levels=10,extend='max')
    format_map(ax,LABELS[name])
fig.colorbar(m,loc='b',label='Diurnal Amplitude of Precipitation (mm)')
axs.format(abc=True,titleloc='l')
pplt.show()

In [ ]:
fig,axs = pplt.subplots(nrows=2,ncols=3,refwidth=2,refheight=1.6,share=False,wspace=3)
for region,color in REGIONS.items():
    for ax,term in zip([axs[0],axs[1],axs[2],axs[4],axs[5]],['M','I','dominance','S','C']):
        ax.plot(hourcenters,termcycles[term,region],color=color,linewidth=1,marker='o',markersize=3)
    axs[3].plot(hourcenters,termcycles['SHF',region],color=color,linewidth=1)
    axs[3].plot(hourcenters,termcycles['LHF',region],color=color,linewidth=1,linestyle='--')
for ax in axs[4:]:
    ax.axhline(0,color='gray',linewidth=0.5)
axs[0].format(title='Moisture Pathway',ylabel='$\\mathit{M}$ (K)')
axs[1].format(title='Instability Pathway',ylabel='$\\mathit{I}$ (K)')
axs[2].format(title='Moisture Dominance',ylabel='Fraction with $\\mathit{M} > \\mathit{I}$')
axs[3].format(title='Surface Fluxes (Solid SHF, Dashed LHF)',ylabel='Flux (W/m$^2$)')
axs[4].format(title='SR-SFC Surface Term',ylabel='$\\mathit{S}$')
axs[5].format(title='SR-ALL-PC Correction Term',ylabel='$\\mathit{C}$')
fig.legend([Line2D([],[],color=color,linewidth=1,label=region) for region,color in REGIONS.items()],loc='b',ncols=2,frame=False)
axs.format(xlabel='Local Solar Time at Window Start (h)',xlim=(0,24),xticks=6,abc=True,titleloc='l')
pplt.show()